Step 30 — Download and verify the pinned Phi-4-mini model

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
python = Path("/opt/b3it-corrected-venv/bin/python")
repo = root / "authors_repo/track-llm-apis"
assert python.is_file() and repo.is_dir(), "Check the project mount and experiment Python."

code = r'''
from pathlib import Path
from collections import defaultdict
import json
import sys

from huggingface_hub import snapshot_download
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer

root = Path(sys.argv[1])
model = "microsoft/Phi-4-mini-instruct"
manifest = json.loads((root / "audit/hub_revisions.json").read_text())
revision = manifest["models"][model]["revision"]
assert revision == "cfbefacb99257ffa30c83adab238a50856ac3083", revision

snapshot = Path(snapshot_download(
    repo_id=model,
    revision=revision,
    cache_dir=str(root / "cache/huggingface/hub"),
    allow_patterns=[
        "*.json", "*.safetensors", "*.txt", "*.model",
        "*.jinja", "*.tiktoken", "*.py",
    ],
))
assert snapshot.name == revision

index = snapshot / "model.safetensors.index.json"
expected = defaultdict(set)
if index.is_file():
    for tensor, shard in json.loads(index.read_text())["weight_map"].items():
        expected[shard].add(tensor)
    assert expected, "Empty checkpoint index"
    shard_names = sorted(expected)
else:
    shard_names = ["model.safetensors"]

sizes = {}
tensor_count = 0
for name in shard_names:
    path = snapshot / name
    assert path.is_file(), f"Missing weight shard: {name}"
    with safe_open(str(path), framework="pt", device="cpu") as shard:
        keys = set(shard.keys())
    assert keys, f"Empty weight shard: {name}"
    if index.is_file():
        assert keys == expected[name], f"Weight names differ from index: {name}"
    tensor_count += len(keys)
    sizes[name] = path.stat().st_size

config = AutoConfig.from_pretrained(snapshot, local_files_only=True, trust_remote_code=False)
tokenizer = AutoTokenizer.from_pretrained(snapshot, local_files_only=True, trust_remote_code=False)
assert tokenizer.encode("Hello", add_special_tokens=False)

record = {
    "model": model,
    "revision": revision,
    "snapshot": str(snapshot),
    "weight_files": sizes,
    "checkpoint_tensor_count": tensor_count,
    "model_type": config.model_type,
    "architectures": config.architectures,
    "tokenizer_class": type(tokenizer).__name__,
}
report = root / "audit/phi4mini_snapshot.json"
if report.exists():
    assert json.loads(report.read_text()) == record, f"Existing audit record differs: {report}"
else:
    tmp = report.with_suffix(".tmp")
    tmp.write_text(json.dumps(record, indent=2) + "\n")
    tmp.replace(report)

print("Revision:", revision)
print("Architecture:", config.architectures)
print("Tokenizer:", type(tokenizer).__name__)
print(f"Weights: {sum(sizes.values()) / 2**30:.2f} GiB across {len(sizes)} shard(s)")
print("Audit:", report)
print("Step 30 passed: pinned Phi-4-mini checkpoint and local tokenizer verified.")
'''

script = root / "scripts/verify_phi4mini_snapshot.py"
script.write_text(code, encoding="utf-8")
started = time.monotonic()
subprocess.run(
    [str(python), "-u", str(script), str(root)],
    cwd=repo,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "HF_HUB_ENABLE_HF_TRANSFER": "0"},
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

Step 31 — Discover Phi-4-mini border inputs

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
python = Path("/opt/b3it-corrected-venv/bin/python")
script = root / "scripts/discover_phi4mini.py"
log = root / "logs/step31_phi4mini_discovery.log"

code = r'''
import json
import sys
from pathlib import Path
from unittest.mock import patch

from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling import bi_phase_1 as phase

def main():
    root = Path(sys.argv[1])
    model = "microsoft/Phi-4-mini-instruct"
    record = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
    snapshot = Path(record["snapshot"])
    assert record["model"] == model
    assert snapshot.name == record["revision"] == "cfbefacb99257ffa30c83adab238a50856ac3083"
    assert config.data_dir.resolve() == (root / "data").resolve()

    bi = config.sampling.bi
    assert (bi.phase_1_tokens_per_endpoint, bi.phase_1_queries_per_token) == (20000, 3)
    assert bi.temperature == 0 and bi.models[model] == 1

    tokenizer = AutoTokenizer.from_pretrained(snapshot, local_files_only=True)
    candidates = phase.get_single_token_inputs(tokenizer, bi.phase_1_tokens_per_endpoint)
    assert len(candidates) == len(set(candidates)) == 20000, len(candidates)

    traffic = phase._load_traffic_prompts()
    assert len(traffic) == 4502, f"Unexpected traffic pool size: {len(traffic)}"

    try:
        result = phase.BIPhase1aOutput.load_model(model)
    except FileNotFoundError:
        result = None

    if result is None:
        original_init = phase.init_vllm_bi

        def init_pinned(model_name, device):
            assert model_name == model
            return original_init(str(snapshot), device)

        print("Starting Phi discovery: 20,000 candidates in one sampler call.", flush=True)
        with patch.object(phase, "init_vllm_bi", init_pinned):
            phase._run_phase(
                output=phase.BIPhase1aOutput(bi_config=bi.model_dump(mode="json")),
                model_inputs={model: candidates},
                model_max_tokens=bi.models,
                n_samples=bi.phase_1_queries_per_token,
                traffic_prompts=traffic,
                device="cuda:0",
            )
        result = phase.BIPhase1aOutput.load_model(model)
    else:
        print("Found saved Phi discovery. Verifying it before reuse.", flush=True)

    assert result.model_name == model and result.phase == "1a"
    assert result.bi_config == bi.model_dump(mode="json"), "Saved discovery settings differ."
    assert set(result.results) == set(candidates), "Saved candidate set differs."
    assert all(sum(counts.values()) == 3 for counts in result.results.values())
    border_inputs = result.get_border_inputs()

    audit = {
        "model": model,
        "revision": record["revision"],
        "candidates": len(candidates),
        "candidate_observations": sum(sum(c.values()) for c in result.results.values()),
        "traffic_prompts": len(traffic),
        "discovered_border_inputs": len(border_inputs),
        "upstream_result": str(result._get_path()),
    }
    path = root / "audit/step31_phi4mini_discovery.json"
    if path.exists():
        assert json.loads(path.read_text()) == audit, f"Existing audit differs: {path}"
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(audit, indent=2) + "\n")
        tmp.replace(path)

    print(json.dumps(audit, indent=2))
    print("Step 31 passed: 20,000 Phi candidates and 60,000 target observations verified.")

if __name__ == "__main__":
    main()
'''

script.write_text(code, encoding="utf-8")
env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        [str(python), "-u", str(script), str(root)],
        cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 31 exited with code {status}. Log: {log}")

Step 32 — Calibrate all 3,072 Phi border inputs

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/calibrate_phi4mini.py"
log = root / "logs/step32_phi4mini_calibration.log"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path
from unittest.mock import patch

from track_llm_apis.config import config
from track_llm_apis.sampling import bi_phase_1 as phase


def main():
    root = Path(sys.argv[1])
    model = "microsoft/Phi-4-mini-instruct"
    snapshot_record = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
    discovery = json.loads((root / "audit/step31_phi4mini_discovery.json").read_text())
    snapshot = Path(snapshot_record["snapshot"])

    assert snapshot.is_dir()
    assert snapshot.name == snapshot_record["revision"] == discovery["revision"]
    assert model == snapshot_record["model"] == discovery["model"]
    assert config.data_dir.resolve() == (root / "data").resolve()

    bi = config.sampling.bi
    assert (bi.phase_1_queries_per_bi, bi.max_border_inputs) == (1000, 120)
    assert (bi.temperature, bi.models[model]) == (0, 1)
    assert (bi.batch_size, bi.per_batch_input_fraction) == (64, 0.95)

    phase_a = phase.BIPhase1aOutput.load_model(model)
    discovered = phase_a.get_border_inputs()
    assert phase_a.bi_config == bi.model_dump(mode="json")
    assert len(discovered) == discovery["discovered_border_inputs"]

    try:
        result = phase.BIPhase1bOutput.load_model(model)
    except FileNotFoundError:
        result = None

    if result is None:
        original_init = phase.init_vllm_bi

        def init_pinned(name, device):
            assert name == model
            return original_init(str(snapshot), device)

        print(f"Calibrating {len(discovered):,} Phi inputs × 1,000 observations.", flush=True)
        with patch.object(phase, "init_vllm_bi", init_pinned):
            phase.phase_1b(device="cuda:0", model=model)
        result = phase.BIPhase1bOutput.load_model(model)
    else:
        print("Found saved Phi calibration. Verifying every input.", flush=True)

    assert result.model_name == model and result.phase == "1b"
    assert result.bi_config == bi.model_dump(mode="json")
    assert set(result.results) == set(discovered), "Calibration and discovery inputs differ."
    assert all(sum(counts.values()) == 1000 for counts in result.results.values())
    assert result.prompt_lengths.keys() == result.results.keys()

    eligible = result.get_border_inputs()
    selected = eligible[:bi.max_border_inputs]
    report = {
        "model": model,
        "revision": snapshot_record["revision"],
        "traffic_prompts": discovery["traffic_prompts"],
        "discovered_inputs": len(discovered),
        "calibration_observations": sum(sum(c.values()) for c in result.results.values()),
        "eligible_after_calibration": len(eligible),
        "selected_count": len(selected),
        "selection_rule": "First 120 returned by phase 1b get_border_inputs()",
        "prompts": selected,
        "prompt_lengths": {p: result.prompt_lengths[p] for p in selected},
        "prompts_sha256": hashlib.sha256(
            json.dumps(selected, ensure_ascii=False).encode("utf-8")
        ).hexdigest(),
        "upstream_result": str(result._get_path()),
    }

    destination = root / "results/phi4mini/selected_probes.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert json.loads(destination.read_text()) == report, "Existing probe selection differs."
    else:
        tmp = destination.with_suffix(".tmp")
        tmp.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
        tmp.replace(destination)

    print(f"Calibration observations verified: {report['calibration_observations']:,}")
    print(f"Eligible after calibration: {len(eligible):,}")
    print(f"Selected probes: {len(selected)}")
    print("Saved:", destination)
    if len(selected) < 120:
        raise RuntimeError(f"Only {len(selected)} probes qualified. Actual results saved.")
    print("Step 32 passed: full Phi calibration verified and 120 ordered probes saved.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
}

started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis",
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 32 exited with code {status}. Log: {log}")

Step 33 — Verify which Phi modules receive LoRA adapters

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/inspect_phi4mini_lora.py"

script.write_text(r'''
import json
import sys
from collections import Counter
from pathlib import Path

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoConfig, AutoModelForCausalLM
from track_llm_apis.tinychange import TinyChangeConfig

root = Path(sys.argv[1])
record = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
snapshot = Path(record["snapshot"])
assert record["model"] == "microsoft/Phi-4-mini-instruct"
assert snapshot.is_dir() and snapshot.name == record["revision"]

cfg = TinyChangeConfig()
model_cfg = AutoConfig.from_pretrained(snapshot, local_files_only=True)
assert (cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout) == (8, 8, 0.0)
assert set(cfg.lora_target_modules) == {"q_proj", "k_proj", "v_proj", "o_proj"}

# Build the real module hierarchy with shape-only tensors for this inspection.
with torch.device("meta"):
    base = AutoModelForCausalLM.from_config(model_cfg, dtype=torch.bfloat16)

fused = [name for name, _ in base.named_modules() if name.endswith(".qkv_proj")]

# Same LoraConfig arguments and PEFT entry point as upstream tinychange.py.
adapter = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.lora_target_modules,
    task_type="CAUSAL_LM",
)
model = get_peft_model(base, adapter)

matched = [
    name for name, module in model.named_modules()
    if "default" in getattr(module, "lora_A", {})
]
trainable = {
    name: list(param.shape)
    for name, param in model.named_parameters()
    if param.requires_grad
}
assert matched and trainable, "PEFT attached no trainable adapters."
expected = {
    f"{name}.lora_{side}.default.weight"
    for name in matched for side in ("A", "B")
}
assert set(trainable) == expected, "Trainable parameters differ from the adapter pairs."

by_projection = dict(Counter(name.rsplit(".", 1)[-1] for name in matched))
report = {
    "model": record["model"],
    "revision": record["revision"],
    "inspection": "Shape-only model before training",
    "configured_targets": cfg.lora_target_modules,
    "rank": cfg.lora_r,
    "alpha": cfg.lora_alpha,
    "dropout": cfg.lora_dropout,
    "fused_qkv_modules": fused,
    "matched_by_projection": by_projection,
    "matched_module_names": matched,
    "trainable_parameter_shapes": trainable,
    "trainable_parameter_count": sum(
        p.numel() for p in model.parameters() if p.requires_grad
    ),
}
path = root / "audit/step33_phi4mini_lora_targets.json"
if path.exists():
    assert json.loads(path.read_text()) == report, f"Existing audit differs: {path}"
else:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(report, indent=2) + "\n")
    tmp.replace(path)

print("Configured targets:", cfg.lora_target_modules)
print("Fused qkv_proj modules in Phi:", len(fused))
print("Actual adapter matches:", by_projection)
print("Trainable parameter count:", f"{report['trainable_parameter_count']:,}")
print("Full module and parameter names:", path)
print("Step 33 passed: Phi LoRA targets inspected and saved before training.")
''', encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "HF_HOME": str(root / "cache/huggingface"),
        "HF_HUB_OFFLINE": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

Step 34 — Verify Phi’s 128 LoRA training examples

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_phi4mini_lora_data.py"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.tinychange import FinetuningDataModule, TinyChangeConfig
from track_llm_apis.util import patch_chat_template

root = Path(sys.argv[1])
snapshot = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
assert snapshot["model"] == "microsoft/Phi-4-mini-instruct"

cfg = TinyChangeConfig()
pool = cfg.finetuning_dataset
assert cfg.datasets_dir.resolve() == (root / "data/datasets").resolve()
assert len(pool) == 4096
assert (cfg.seed, cfg.finetuning_batch_size, cfg.finetuning_epochs) == (0, 1, 1)

tokenizer = AutoTokenizer.from_pretrained(snapshot["snapshot"], local_files_only=True)
tokenizer.name_or_path = snapshot["model"]
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
patch_chat_template(tokenizer, config.chat_templates)

dm = FinetuningDataModule(
    config=cfg, n_samples=128, batch_size=cfg.finetuning_batch_size, tokenizer=tokenizer,
)
indices = np.random.RandomState(cfg.seed).permutation(len(pool))[:128].tolist()
np.random.seed(cfg.seed)
dm.prepare_data()
dm.setup("fit")

loader = dm.train_dataloader()
assert len(loader) == 128
target_counts = []
padded_lengths = set()

for position, (batch, pool_index) in enumerate(zip(loader, indices, strict=True)):
    raw = pool[pool_index]["conversation"]
    expected = tokenizer.apply_chat_template(
        raw, tokenize=True, return_dict=True, return_assistant_tokens_mask=True,
        padding=False, truncation=False,
    )
    ids = batch["input_ids"][0].tolist()
    attention = batch["attention_mask"][0].tolist()
    assistant = batch["assistant_tokens_mask"][0].tolist()
    assert len(ids) == len(attention) == len(assistant), position
    assert batch["labels"][0].tolist() == ids, position
    assert [v for v, keep in zip(ids, attention) if keep] == expected["input_ids"], position
    assert [v for v, keep in zip(assistant, attention) if keep] == expected["assistant_masks"], position
    assert all(a == 0 for a, keep in zip(assistant, attention) if not keep), position
    padded_lengths.add(len(ids))
    target_counts.append(sum(assistant[1:]))

assert len(padded_lengths) == 1, f"Observed padded lengths: {sorted(padded_lengths)}"
padded_length = padded_lengths.pop()
empty = [i for i, count in enumerate(target_counts) if count == 0]
assert not empty, f"Training positions with zero assistant targets: {empty}"

report = {
    "model": snapshot["model"],
    "revision": snapshot["revision"],
    "training_pool_size": len(pool),
    "training_pool_fingerprint": pool._fingerprint,
    "seed": cfg.seed,
    "selected_examples": len(indices),
    "selected_pool_indices_in_training_order": indices,
    "indices_sha256": hashlib.sha256(json.dumps(indices).encode()).hexdigest(),
    "configured_max_length": cfg.finetuning_max_length,
    "padded_tokens_per_example": padded_length,
    "minimum_trainable_assistant_tokens": min(target_counts),
    "data_module_cache": str(dm._get_disk_path()),
}
path = root / "audit/step34_phi4mini_lora_data.json"
if path.exists():
    assert json.loads(path.read_text()) == report, f"Existing audit differs: {path}"
else:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(report, indent=2) + "\n")
    tmp.replace(path)

print("Verified examples in training order:", len(indices))
print("Configured max length:", cfg.finetuning_max_length)
print("Actual padded tokens per example:", padded_length)
print("Minimum trainable assistant tokens:", min(target_counts))
print("Saved:", path)
print("Step 34 passed: Phi LoRA training data, order, and masks verified.")
''', encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "HF_HOME": str(root / "cache/huggingface"),
        "HF_HUB_OFFLINE": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

Step 35 — Train and verify the Phi LoRA checkpoint

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
python = Path("/opt/b3it-corrected-venv/bin/python")
script = root / "scripts/train_phi4mini_lora.py"
log = root / "logs/step35_phi4mini_lora.log"

# Reuse the successful version, including its tensor-by-tensor reload check.
source = (root / "scripts/train_qwen_lora.py").read_text(encoding="utf-8")
assert 'record["verified_checkpoint_tensors"]' in source, "Expected the corrected training script."
assert 'saved["reloaded_hash"]' in source, "Expected the corrected resume check."

for old, new in [
    ("step09_qwen_lora_data.json", "step34_phi4mini_lora_data.json"),
    ("qwen_snapshot.json", "phi4mini_snapshot.json"),
    ("results/qwen/checkpoints", "results/phi4mini/checkpoints"),
    (".qwen-lora-", ".phi4mini-lora-"),
    ("Step 10 passed", "Step 35 passed"),
    ("Qwen LoRA", "Phi LoRA"),
]:
    assert old in source, f"Training script layout differs at {old!r}."
    source = source.replace(old, new)

marker = '        record["trainable_parameters"] = trainable'
assert source.count(marker) == 1
source = source.replace(marker, '''        audit = json.loads(
            (root / "audit/step33_phi4mini_lora_targets.json").read_text()
        )
        actual_shapes = {
            name: list(p.shape)
            for name, p in model.model.named_parameters() if p.requires_grad
        }
        assert audit["revision"] == snapshot["revision"]
        assert actual_shapes == audit["trainable_parameter_shapes"], "LoRA targets differ from Step 33"
        assert audit["matched_by_projection"] == {"o_proj": 32}
        count = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        assert count == audit["trainable_parameter_count"] == 1572864
        record["trainable_parameter_count"] = count
        record["matched_by_projection"] = audit["matched_by_projection"]
        print("Verified before training: 32 o_proj adapters, 1,572,864 trainable parameters.", flush=True)
''' + marker)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text(encoding="utf-8") == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.free,memory.total", "--format=csv"],
    check=True,
)

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        [str(python), "-u", str(script), str(root)],
        cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 35 exited with code {status}. Log: {log}")

import json
checkpoint = root / "results/phi4mini/checkpoints/lora_128"
report = json.loads((checkpoint / "COMPLETE.json").read_text())
assert report["optimizer_steps"] == 128
assert report["merged_dtype"] == "torch.bfloat16"
assert report["verified_checkpoint_tensors"] > 0
print("BF16 parameter values changed:", report["changed_bf16_parameter_values"])
print("Peak PyTorch allocation (GiB):", report["peak_torch_allocated_gib"])
print("Verified checkpoint:", checkpoint)

Step 36 — Create Phi’s three remaining perturbations

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
python = Path("/opt/b3it-corrected-venv/bin/python")
checkpoints = root / "results/phi4mini/checkpoints"
script = root / "scripts/create_phi4mini_weight_variants.py"
log = root / "logs/step36_phi4mini_weight_variants.log"

snapshot = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
lora = json.loads((checkpoints / "lora_128/COMPLETE.json").read_text())
assert snapshot["model"] == lora["model"] == "microsoft/Phi-4-mini-instruct"
assert snapshot["revision"] == lora["revision"]
assert lora["optimizer_steps"] == 128

# Use the same creation, save, reload, and resume logic as Step 11.
source = (root / "scripts/create_qwen_weight_variants.py").read_text()
for old, new in [
    ("audit/qwen_snapshot.json", "audit/phi4mini_snapshot.json"),
    ("results/qwen/checkpoints", "results/phi4mini/checkpoints"),
    ("Qwen", "Phi"),
    ("Step 11", "Step 36"),
]:
    assert old in source, f"Unexpected Step 11 script layout: {old!r}"
    source = source.replace(old, new)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text() == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

names = ("noise", "magnitude", "random")
remaining = [name for name in names if not (checkpoints / name / "COMPLETE.json").is_file()]
weight_bytes = sum(snapshot["weight_files"].values())
print("Conditions awaiting creation:", remaining, flush=True)
print(f"Approximate additional weights: {len(remaining) * weight_bytes / 2**30:.2f} GiB", flush=True)

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        [str(python), "-u", str(script), str(root)],
        cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

if status:
    raise RuntimeError(f"Step 36 exited with code {status}. Log: {log}")

for name in names:
    report = json.loads((checkpoints / name / "COMPLETE.json").read_text())
    assert report["revision"] == snapshot["revision"]
    assert report["base_hash"] == lora["base_hash"]
    assert report["creation_dtype"] == "torch.bfloat16"
    assert report["verified_checkpoint_tensors"] == lora["verified_checkpoint_tensors"]
    print(f"{name}: {report['changed_bf16_parameter_values']:,} changed BF16 values")

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print("Step 36 verified: all three Phi perturbation reports match the Step 35 baseline.")

Step 37 — Verify Phi’s evaluation inputs and save the manifest

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
results = root / "results/phi4mini"
script = root / "scripts/prepare_phi4mini_evaluation.py"

# Reuse the corrected Step 12 script that verified the ordered traffic prompts.
source = (root / "scripts/prepare_qwen_evaluation.py").read_text(encoding="utf-8")
for old, new in [
    ('"results/qwen"', '"results/phi4mini"'),
    ("audit/qwen_snapshot.json", "audit/phi4mini_snapshot.json"),
    ("Step 12 passed", "Step 37 passed"),
]:
    assert old in source, f"Unexpected evaluation script layout: {old!r}"
    source = source.replace(old, new)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text(encoding="utf-8") == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=repo,
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
        "HF_HOME": str(root / "cache/huggingface"),
        "HF_HUB_OFFLINE": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
    },
    check=True,
)

path = results / "evaluation_manifest.json"
manifest = json.loads(path.read_text(encoding="utf-8"))
selection = json.loads((results / "selected_probes.json").read_text(encoding="utf-8"))
assert manifest["model"] == "microsoft/Phi-4-mini-instruct"
assert manifest["revision"] == selection["revision"] == "cfbefacb99257ffa30c83adab238a50856ac3083"
assert manifest["probes"] == selection["prompts"]
assert len(manifest["probes"]) == len(set(manifest["probes"])) == 120
assert [c["id"] for c in manifest["conditions"]] == [
    "unchanged", "lora_128", "noise", "magnitude", "random",
]
assert manifest["observations_per_probe_per_condition"] == 5000
assert manifest["planned_target_observations"] == 3_000_000

print("Manifest:", path)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print("Step 37 verified: 120 ordered Phi probes, five conditions, 3,000,000 observations planned.")

Step 38 — Collect the unchanged Phi baseline

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import tempfile
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
results = root / "results/phi4mini"
script = root / "scripts/collect_phi4mini_baseline.py"
log = root / "logs/step38_phi4mini_baseline.log"

manifest = json.loads((results / "evaluation_manifest.json").read_text())
assert manifest["model"] == "microsoft/Phi-4-mini-instruct"
assert manifest["revision"] == "cfbefacb99257ffa30c83adab238a50856ac3083"
assert len(manifest["probes"]) == 120
assert manifest["observations_per_probe_per_condition"] == 5000

# The upstream engine writes a temporary model copy during initialization.
output = results / "observations/unchanged.json"
if not output.exists():
    snapshot = json.loads((root / "audit/phi4mini_snapshot.json").read_text())
    weights = sum(snapshot["weight_files"].values())
    temp_dir = tempfile.gettempdir()
    free = shutil.disk_usage(temp_dir).free
    print(f"Temporary disk {temp_dir}: {free / 2**30:.1f} GiB free", flush=True)
    assert free > weights, "Temporary disk has insufficient room for the Phi model copy."

source = (root / "scripts/collect_qwen_baseline.py").read_text(encoding="utf-8")
for old, new in [
    ('"results/qwen"', '"results/phi4mini"'),
    ("unchanged Qwen", "unchanged Phi"),
    ("Step 13 passed", "Step 38 passed"),
]:
    assert old in source, f"Unexpected baseline script layout: {old!r}"
    source = source.replace(old, new)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text(encoding="utf-8") == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 38 exited with code {status}. Log: {log}")

saved = json.loads(output.read_text())
assert saved["model"] == manifest["model"] and saved["revision"] == manifest["revision"]
assert saved["condition"] == "unchanged" and saved["inference_dtype"] == "torch.bfloat16"
assert set(saved["counts"]) == set(manifest["probes"])
assert all(sum(counts.values()) == 5000 for counts in saved["counts"].values())
print("Saved:", output)
print("Step 38 verified: 600,000 fresh unchanged-Phi observations.")

In [ ]:
Step 39 — Collect Phi observations for the four changed conditions

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo/track-llm-apis"
scripts = root / "scripts"
results = root / "results/phi4mini"
script = scripts / "collect_phi4mini_changed.py"

assert (results / "observations/unchanged.json").is_file(), "Missing Step 38 baseline."
assert (scripts / "collect_phi4mini_baseline.py").is_file(), "Missing Step 38 verifier."

source = (scripts / "collect_gemma_changed.py").read_text(encoding="utf-8")
for old, new in [
    ("from collect_gemma_baseline import", "from collect_phi4mini_baseline import"),
    ('"results/gemma"', '"results/phi4mini"'),
    ('"google/gemma-3-1b-it"', '"microsoft/Phi-4-mini-instruct"'),
]:
    assert source.count(old) == 1, f"Unexpected collector layout at {old!r}"
    source = source.replace(old, new)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text(encoding="utf-8") == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "CUDA_VISIBLE_DEVICES": "0",
    "MPLBACKEND": "Agg",
}

started = time.monotonic()
for name in ("lora_128", "noise", "magnitude", "random"):
    log = root / f"logs/step39_phi4mini_{name}.log"
    print(f"\nStarting {name}. Log: {log}", flush=True)
    with log.open("a", encoding="utf-8") as out:
        with subprocess.Popen(
            ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root), name],
            cwd=repo, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    out.write(line)
                    out.flush()
                status = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                process.wait()
                raise
    if status:
        raise RuntimeError(f"{name} exited with code {status}. See {log}")

print("Step 39 passed: 3,000,000 Phi evaluation observations verified across five conditions.")
print("Saved in:", results / "observations")
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

In [ ]:
Step 40 — Prepare Phi’s scoring input on CPU

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
results = root / "results/phi4mini"
scripts = root / "scripts"
script = scripts / "prepare_phi4mini_scoring_input.py"

manifest = json.loads((results / "evaluation_manifest.json").read_text())
assert manifest["model"] == "microsoft/Phi-4-mini-instruct"
assert manifest["planned_target_observations"] == 3_000_000

source = (scripts / "prepare_gemma_scoring_input.py").read_text(encoding="utf-8")
for old, new in [
    ('"results/gemma"', '"results/phi4mini"'),
    (".gemma-merge-", ".phi4mini-merge-"),
    ("Step 27 passed: 3,000,000 Gemma", "Step 40 passed: 3,000,000 Phi"),
]:
    assert old in source, f"Unexpected Step 27 script layout at {old!r}"
    source = source.replace(old, new)

compile(source, str(script), "exec")
if script.exists():
    assert script.read_text(encoding="utf-8") == source, f"Existing script differs: {script}"
else:
    script.write_text(source, encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg"},
    check=True,
)

archive = results / "figure1_merged/microsoft2fPhi-4-mini-instruct.json.gz"
assert archive.is_file() and archive.stat().st_size > 0, f"Missing Phi archive: {archive}"
print("Phi scoring input:", archive)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

In [ ]:
Step 41 — Score Phi with B3IT on CPU

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

root = Path("/workspace/B3IT_Figure1_Corrected")
results = root / "results/phi4mini"
scripts = root / "scripts"
archive = results / "figure1_merged/microsoft2fPhi-4-mini-instruct.json.gz"
assert archive.is_file() and archive.stat().st_size > 0, f"Missing Step 40 archive: {archive}"

manifest = json.loads((results / "evaluation_manifest.json").read_text())
assert manifest["model"] == "microsoft/Phi-4-mini-instruct"
assert manifest["reference_samples_per_prompt"] == 50
assert manifest["observations_per_probe_per_condition"] == 5000

source = (scripts / "run_gemma_b3it_step28.py").read_text(encoding="utf-8")
# Replace full model identifiers before replacing project folder and script names.
for old, new in [
    ("google/gemma-3-1b-it", "microsoft/Phi-4-mini-instruct"),
    ("google2fgemma-3-1b-it", "microsoft2fPhi-4-mini-instruct"),
    ("gemma", "phi4mini"),
    ("Gemma", "Phi"),
    ("GEMMA", "PHI4MINI"),
]:
    source = source.replace(old, new)

source = re.sub(r"(?i)\bstep([ _-]?)28\b", r"Step\g<1>41", source)
assert "results/phi4mini" in source, "Check the scoring input directory."
assert "gemma" not in source.lower(), "A Gemma reference remains."

runner = scripts / "run_phi4mini_b3it_step41.py"
compile(source, str(runner), "exec")
if runner.exists():
    assert runner.read_text(encoding="utf-8") == source, f"Existing runner differs: {runner}"
else:
    runner.write_text(source, encoding="utf-8")

print("Scoring input:", archive, flush=True)
subprocess.run(
    [sys.executable, "-u", str(runner)],
    cwd=root / "authors_repo/track-llm-apis",
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg"},
    check=True,
)

stats = results / "figure1_merged/analysis/bi"
assert stats.is_dir() and any(stats.iterdir()), f"Missing Phi statistics: {stats}"
print("Phi statistics saved:", stats)

Step 42 — Export and verify Phi’s Figure 1 CSVs

In [ ]:
from pathlib import Path
from collections import defaultdict
from statistics import fmean
import csv
import json
import math
import os
import re
import subprocess
import sys
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
scripts = root / "scripts"
results = root / "results/phi4mini"
model = "microsoft/Phi-4-mini-instruct"
stats = results / "figure1_merged/analysis/bi"
assert stats.is_dir() and any(stats.iterdir()), f"Missing Step 41 statistics: {stats}"

source = (scripts / "run_gemma_export_step29.py").read_text(encoding="utf-8")
for old, new in [
    ("google/gemma-3-1b-it", model),
    ("google2fgemma-3-1b-it", "microsoft2fPhi-4-mini-instruct"),
    ("gemma", "phi4mini"),
    ("Gemma", "Phi"),
    ("GEMMA", "PHI4MINI"),
]:
    source = source.replace(old, new)
source = re.sub(r"(?i)\bstep([ _-]?)29\b", r"step\g<1>42", source)
assert "results/phi4mini" in source and "gemma" not in source.lower()

runner = scripts / "run_phi4mini_export_step42.py"
compile(source, str(runner), "exec")
if runner.exists():
    assert runner.read_text(encoding="utf-8") == source, f"Existing runner differs: {runner}"
else:
    runner.write_text(source, encoding="utf-8")

started = time.monotonic()
subprocess.run(
    [sys.executable, "-u", str(runner)],
    cwd=root / "authors_repo/track-llm-apis",
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg"},
    check=True,
)

def read_csv(path):
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))

def setting(row):
    return int(row["n_prompts"]), int(row["samples_per_prompt"])

def variant_key(description):
    return json.dumps(description, sort_keys=True)

out = results / "figure1"
per_variant = out / "figure1_per_variant.csv"
summary = out / "figure1_summary.csv"
rows, means = read_csv(per_variant), read_csv(summary)
manifest = json.loads((results / "evaluation_manifest.json").read_text())
expected_variants = {
    variant_key(c["description"])
    for c in manifest["conditions"] if c["id"] != "unchanged"
}
grid = {(p, s) for p in (1, 3, 5, 10) for s in (1, 3, 10, 20, 50)}

assert len(expected_variants) == 4
assert len(rows) == 80 and len(means) == 20
assert {row["model"] for row in rows} == {model}
assert {setting(row) for row in rows} == {setting(row) for row in means} == grid

grouped = defaultdict(dict)
for row in rows:
    key = setting(row)
    variant = variant_key(json.loads(row["variant"]))
    auc = float(row["roc_auc"])
    cost = float(row["yearly_cost_usd"])
    assert variant not in grouped[key], f"Duplicate variant at {key}"
    assert 0 <= auc <= 1 and math.isfinite(cost) and cost > 0, row
    grouped[key][variant] = (auc, cost)

for row in means:
    key = setting(row)
    values = grouped[key]
    assert set(values) == expected_variants, f"Variant mismatch at {key}"
    assert int(row["number_of_variants"]) == 4
    assert math.isclose(float(row["mean_roc_auc"]),
                        fmean(a for a, _ in values.values()), abs_tol=1e-9), key
    assert math.isclose(float(row["yearly_cost_usd"]),
                        fmean(c for _, c in values.values()), abs_tol=1e-9), key

lowest = min(rows, key=lambda row: float(row["roc_auc"]))
print(f"Lowest Phi AUC: {float(lowest['roc_auc']):.6f}")
print("Perturbation:", lowest["variant"])
print("Setting (prompts, detection samples per prompt):", setting(lowest))
print("Per-variant CSV:", per_variant)
print("Summary CSV:", summary)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print("Step 42 verified: 80 Phi AUC rows and 20 equal-weight four-variant means.")

Step 43 — Combine the three models and verify the 20 Figure 1 points

In [ ]:
from pathlib import Path
from collections import defaultdict
from statistics import fmean
import csv
import json
import math

root = Path("/workspace/B3IT_Figure1_Corrected")
out = root / "results/figure1_combined"
models = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "gemma": "google/gemma-3-1b-it",
    "phi4mini": "microsoft/Phi-4-mini-instruct",
}
grid = {(p, s) for p in (1, 3, 5, 10) for s in (1, 3, 10, 20, 50)}

def variant_key(description):
    return json.dumps(description, sort_keys=True, separators=(",", ":"))

variants = {
    variant_key({"type": "finetune", "lr": 1e-6, "n_samples": 128, "lora": True}),
    variant_key({"type": "random_noise", "scale": 2**-12}),
    variant_key({"type": "weight_pruning", "method": "magnitude", "scale": 2**-8}),
    variant_key({"type": "weight_pruning", "method": "random", "scale": 2**-8}),
}

def read_csv(path):
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))

def setting(row):
    return int(row["n_prompts"]), int(row["samples_per_prompt"])

pairs = []
by_setting = defaultdict(list)

for folder, model in models.items():
    directory = root / "results" / folder / "figure1"
    rows = read_csv(directory / "figure1_per_variant.csv")
    summaries = read_csv(directory / "figure1_summary.csv")
    assert len(rows) == 80 and len(summaries) == 20, model
    assert {setting(row) for row in summaries} == grid, model

    local = defaultdict(dict)
    for row in rows:
        key = setting(row)
        variant = variant_key(json.loads(row["variant"]))
        auc = float(row["roc_auc"])
        cost = float(row["yearly_cost_usd"])
        assert row["model"] == model and key in grid, (model, key)
        assert variant in variants and variant not in local[key], (model, key, variant)
        assert 0 <= auc <= 1 and math.isfinite(cost) and cost > 0, row

        entry = {
            "model": model, "variant": variant, "n_prompts": key[0],
            "samples_per_prompt": key[1], "roc_auc": auc, "yearly_cost_usd": cost,
        }
        local[key][variant] = entry
        pairs.append(entry)
        by_setting[key].append(entry)

    # Independently reproduce each previously verified model summary.
    for row in summaries:
        key = setting(row)
        assert set(local[key]) == variants, (model, key)
        assert int(row["number_of_variants"]) == 4
        for source, target in [("roc_auc", "mean_roc_auc"),
                               ("yearly_cost_usd", "yearly_cost_usd")]:
            calculated = fmean(v[source] for v in local[key].values())
            assert math.isclose(calculated, float(row[target]), abs_tol=1e-9), (
                model, key, target
            )
    print(f"{model}: 80 unique AUCs and 20 model means verified.")

summary = []
expected_pairs = {(model, variant) for model in models.values() for variant in variants}
for p, s in sorted(grid):
    group = by_setting[p, s]
    assert len(group) == 12
    assert {(r["model"], r["variant"]) for r in group} == expected_pairs
    summary.append({
        "n_prompts": p,
        "samples_per_prompt": s,
        "number_of_model_variant_pairs": 12,
        "mean_roc_auc": fmean(r["roc_auc"] for r in group),
        "yearly_cost_usd": fmean(r["yearly_cost_usd"] for r in group),
    })

out.mkdir(parents=True, exist_ok=True)
def save_csv(name, rows):
    path = out / name
    tmp = path.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    tmp.replace(path)
    print("Saved:", path)

save_csv("figure1_three_models_per_variant.csv", pairs)
save_csv("figure1_three_models_summary.csv", summary)

for point in (summary[0], summary[-1]):
    print(f"{point['n_prompts']} prompt(s) × {point['samples_per_prompt']} sample(s): "
          f"mean AUC {point['mean_roc_auc']:.6f}, annual cost ${point['yearly_cost_usd']:.4f}")
print("Step 43 passed: 240 model–variant AUCs verified; 20 equal-weight points saved.")

Step 44 — Calculate the authors’ 95% bootstrap intervals

In [ ]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/bootstrap_three_models.py"

code = r'''
import csv
import hashlib
import json
import math
import multiprocessing as mp
import os
import random
import sys
import time
from pathlib import Path
from statistics import fmean

from track_llm_apis.config import config
from track_llm_apis.sampling import analyze_param_sweeps as sweep

root = Path(sys.argv[1])
out = root / "results/figure1_combined"
cache = out / "bootstrap"
cache.mkdir(exist_ok=True)

def read_csv(path):
    with path.open(encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

def key(row):
    return int(row["n_prompts"]), int(row["samples_per_prompt"])

def pair_id(model, variant):
    return model, json.dumps(json.loads(variant), sort_keys=True, separators=(",", ":"))

# Linux fork workers inherit the current setting's score arrays read-only.
# Each task passes only replicate indices between processes.
def bootstrap_batch(bounds):
    start, stop = bounds
    values = []
    for index in range(start, stop):
        random.seed(seed_base + index)
        values.append(float(sweep._bootstrap_one_hierarchical(score_pairs)))
    return values

if __name__ == "__main__":
    summaries = read_csv(out / "figure1_three_models_summary.csv")
    pairs = read_csv(out / "figure1_three_models_per_variant.csv")
    grid = {(p, s) for p in (1, 3, 5, 10) for s in (1, 3, 10, 20, 50)}
    assert len(summaries) == 20 and {key(r) for r in summaries} == grid
    assert len(pairs) == 240

    expected = {k: {} for k in grid}
    for row in pairs:
        k = key(row)
        pair = pair_id(row["model"], row["variant"])
        assert pair not in expected[k]
        expected[k][pair] = float(row["roc_auc"])

    directories = [
        root / "results" / folder / "figure1_merged"
        for folder in ("qwen", "gemma", "phi4mini")
    ]
    raw = sweep.load_single_variant_results(directories, sweep.BIParameterSweep.get_data_source())
    grouped = {k: {} for k in grid}
    for result in raw:
        k = (result.params["n_prompts"], result.params["detection_samples"])
        pair = pair_id(result.model_name, result.variant)
        assert pair in expected[k] and pair not in grouped[k], (k, pair)
        h0, h1 = result.h0_stats.stats, result.h1_stats.stats
        assert (len(h0), len(h1)) == (2000, 1000), (k, pair)
        auc = sweep.compute_auc_from_stats(h0, h1, sampling=False)
        assert math.isclose(auc, expected[k][pair], abs_tol=1e-9), (k, pair)
        grouped[k][pair] = (h0, h1)

    n = config.analysis.n_bootstrap
    alpha = config.analysis.results_alpha
    assert n == 10000 and alpha == 0.05
    workers = min(9, len(os.sched_getaffinity(0)))
    print(f"Verified score files. Using {workers} CPU workers and {n:,} replicates per setting.",
          flush=True)

    completed = []
    new_times = []
    for setting_index, row in enumerate(summaries):
        p, s = k = key(row)
        assert len(grouped[k]) == 12 and grouped[k].keys() == expected[k].keys()
        assert math.isclose(fmean(expected[k].values()), float(row["mean_roc_auc"]),
                            abs_tol=1e-9)
        ordered_pairs = sorted(grouped[k])
        score_pairs = [grouped[k][pair] for pair in ordered_pairs]
        seed_base = (p * 100 + s) * n
        fingerprint = hashlib.sha256(
            json.dumps([ordered_pairs, score_pairs, seed_base, n, alpha],
                       separators=(",", ":")).encode()
        ).hexdigest()
        path = cache / f"prompts_{p}_samples_{s}.json"

        if path.exists():
            saved = json.loads(path.read_text())
            assert saved["fingerprint"] == fingerprint, f"Saved inputs differ: {path}"
            values = saved["bootstrap_aucs"]
            status = "reused"
        else:
            started = time.monotonic()
            batches = [(i, min(i + 100, n)) for i in range(0, n, 100)]
            with mp.get_context("fork").Pool(workers) as pool:
                values = [v for batch in pool.imap(bootstrap_batch, batches) for v in batch]
            assert len(values) == n and all(0 <= v <= 1 for v in values)
            tmp = path.with_suffix(".tmp")
            tmp.write_text(json.dumps({"fingerprint": fingerprint, "bootstrap_aucs": values}))
            tmp.replace(path)
            new_times.append(time.monotonic() - started)
            status = "saved"

        assert len(values) == n and all(0 <= v <= 1 for v in values)
        lower, upper = sweep.ci(values, alpha)
        completed.append(dict(row, auc_lower=lower, auc_upper=upper, n_bootstrap=n))
        remaining = len(summaries) - setting_index - 1
        eta = f" | rough remaining {remaining * fmean(new_times) / 60:.1f} min" if new_times else ""
        print(f"{setting_index + 1:2d}/20 | {p} prompts × {s} samples | "
              f"95% interval [{lower:.6f}, {upper:.6f}] | {status}{eta}", flush=True)

    destination = out / "figure1_three_models_summary_with_ci.csv"
    tmp = destination.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(completed[0]))
        writer.writeheader()
        writer.writerows(completed)
    tmp.replace(destination)
    assert len(read_csv(destination)) == 20
    print("Saved:", destination)
    print("Step 44 passed: 20 bootstrap intervals saved using 10,000 replicates each.")
'''

script.write_text(code, encoding="utf-8")
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg",
        "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

Step 45 — Generate the combined Figure 1 PNG and PDF

In [ ]:
from pathlib import Path
import os
import subprocess
from IPython.display import Image, display

root = Path("/workspace/B3IT_Figure1_Corrected")
out = root / "results/figure1_combined"
script = root / "scripts/plot_three_models_figure1.py"

script.write_text(r'''
import csv
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter

out = Path(sys.argv[1])
source = out / "figure1_three_models_summary_with_ci.csv"
with source.open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

prompts = [1, 3, 5, 10]
samples = [1, 3, 10, 20, 50]
points = {}
for row in rows:
    key = int(row["n_prompts"]), int(row["samples_per_prompt"])
    assert key not in points, f"Duplicate setting: {key}"
    assert int(row["number_of_model_variant_pairs"]) == 12
    assert int(row["n_bootstrap"]) == 10000
    cost = float(row["yearly_cost_usd"])
    auc = float(row["mean_roc_auc"])
    lo, hi = float(row["auc_lower"]), float(row["auc_upper"])
    assert math.isfinite(cost) and cost > 0 and 0 <= auc <= 1
    assert 0 <= lo <= hi <= 1
    points[key] = cost, auc

assert set(points) == {(p, s) for p in prompts for s in samples}

title = "Scaled reproduction of Figure 1: three models, four perturbation variants per model."
colors = {p: plt.get_cmap("turbo")(i / 3) for i, p in enumerate(prompts)}
diameters = {s: 8 + 2 * i for i, s in enumerate(samples)}

fig = plt.figure(figsize=(10.5, 5.3), layout="constrained")
grid = fig.add_gridspec(1, 2, width_ratios=[4, 1.25])
ax = fig.add_subplot(grid[0, 0])
legend_ax = fig.add_subplot(grid[0, 1])
legend_ax.axis("off")
fig.suptitle(title.replace(": ", ":\n", 1), fontsize=14)

for p in prompts:
    ordered = sorted((points[p, s][0], points[p, s][1], s) for s in samples)
    costs, aucs, sample_counts = zip(*ordered)
    ax.plot(costs, aucs, color=colors[p], linewidth=1.8, zorder=2)
    ax.scatter(costs, aucs, s=[diameters[s] ** 2 for s in sample_counts],
               color=colors[p], linewidths=0, zorder=3)

ax.set_xscale("log")
ax.set_xlim(0.1, 100)
ax.xaxis.set_major_locator(FixedLocator([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]))
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value:g}"))
ax.set_xlabel("Cost per year ($)")
ax.set_ylabel("Average ROC AUC")
ax.margins(y=0.12)
ax.grid(True, which="major", alpha=0.22)

color_handles = [
    Line2D([], [], color=colors[p], marker="o", markersize=5, label=str(p))
    for p in prompts
]
size_handles = [
    Line2D([], [], linestyle="none", marker="o", markersize=diameters[s],
           markerfacecolor="none", markeredgecolor="black", label=str(s))
    for s in samples
]
first = legend_ax.legend(handles=color_handles, title="Prompts per test",
                         loc="upper left", frameon=False)
legend_ax.add_artist(first)
legend_ax.legend(handles=size_handles, title="Detection samples\nper prompt",
                 loc="lower left", frameon=False, labelspacing=0.9)

for ext in ("png", "pdf"):
    path = out / f"figure1_three_models.{ext}"
    tmp = path.with_name(f".{path.stem}.tmp.{ext}")
    fig.savefig(tmp, format=ext, dpi=300, bbox_inches="tight", facecolor="white")
    with tmp.open("rb") as f:
        signature = f.read(8)
    assert signature.startswith(b"%PDF-" if ext == "pdf" else b"\x89PNG\r\n\x1a\n")
    tmp.replace(path)
    print(f"{ext.upper()}: {path}")

plt.close(fig)
print("Combined per-variant data:", out / "figure1_three_models_per_variant.csv")
print("Combined point estimates:", out / "figure1_three_models_summary.csv")
print("Combined estimates and intervals:", source)
print("Step 45 passed: combined Figure 1 saved as PNG and PDF.")
''', encoding="utf-8")

subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(out)],
    cwd=root,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg"},
    check=True,
)
display(Image(filename=str(out / "figure1_three_models.png"), width=950))